In [1]:
import pandas as pd
import numpy as np
from grape import EmbeddingResult
import pickle

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load embeddings from DGLKE results as GRAPE embeddings and save

### Load mapping files

In [2]:
entity_map_file = '../data/npkg/entities.tsv'
relation_map_file = '../data/npkg/relations.tsv'

In [3]:
entities_df = pd.read_csv(entity_map_file, sep=',', header=None, names=['id', 'entity'])
entities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1089125 entries, 0 to 1089124
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   id      1089125 non-null  int64 
 1   entity  1089125 non-null  object
dtypes: int64(1), object(1)
memory usage: 16.6+ MB


In [4]:
relations_df = pd.read_csv(relation_map_file, sep=',', header=None, names=['id', 'relation'])
relations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1343 entries, 0 to 1342
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        1343 non-null   int64 
 1   relation  1343 non-null   object
dtypes: int64(1), object(1)
memory usage: 21.1+ KB


In [5]:
node_mapping = {row['id']: row['entity'] for _, row in entities_df.iterrows()}
edge_type_mapping = {row['id']: row['relation'] for _, row in relations_df.iterrows()}

### Load embeddings

In [6]:
EMBED_DIR = '../dglke-modeling/output/'

In [7]:
def load_embeddings(model_name):
    print('Loading embeddings for model:', model_name)
    EMBED_PATH = EMBED_DIR + model_name + '_npkg_embeddings_all_data/'
    EMBED_ENTITY_FILENAME = 'npkg_'+model_name+'_entity.npy'
    EMBED_RELATION_FILENAME = 'npkg_'+model_name+'_relation.npy'
    entity_embedding = np.load(EMBED_PATH + EMBED_ENTITY_FILENAME)
    relation_embedding = np.load(EMBED_PATH + EMBED_RELATION_FILENAME)
    print('Embeddings loaded')
    return entity_embedding, relation_embedding

In [8]:
def create_df_from_embeddings(entity_embedding, relation_embedding):
    print('Processing embeddings to dataframe')
    node_embeddings_df = pd.DataFrame(entity_embedding, index=entities_df['entity'])
    edge_type_embeddings_df = pd.DataFrame(relation_embedding, index=relations_df['relation'])
    return node_embeddings_df, edge_type_embeddings_df

In [9]:
def get_embedding_result(model_name):
    entity_embedding, relation_embedding = load_embeddings(model_name)
    node_embeddings_df, edge_type_embeddings_df = create_df_from_embeddings(entity_embedding, relation_embedding)
    embedding_result = EmbeddingResult(
        embedding_method_name=model_name,
        node_embeddings=node_embeddings_df,
        edge_type_embeddings=edge_type_embeddings_df
    )
    print('Embedding result created')
    return embedding_result

In [10]:
##change model name before running
model_name = 'TransR'
embedding_result = get_embedding_result(model_name)
embedding_result_dict = embedding_result.dump()
with open('embeddings/npkg_'+model_name+'_embeddings.pickle', 'wb') as f:
    pickle.dump(embedding_result_dict, f)

Loading embeddings for model: TransR
Embeddings loaded
Processing embeddings to dataframe
Embedding result created


In [2]:
##load and check embedding result
model_name = 'TransR'
with open('embeddings/npkg_'+model_name+'_embeddings.pickle', 'rb') as f:
    embedding_result_dict = pickle.load(f)
embedding_result = EmbeddingResult.load(embedding_result_dict)
embedding_result.get_node_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
entity,,,,,,,,,,,,,,,,,,,,,
clo:0035452,0.172132,-0.135239,0.011452,-0.024510,0.076186,0.158029,-0.073178,0.136557,0.083828,-0.145550,...,-0.296592,-0.182115,-0.142333,0.118997,-0.078996,0.215671,0.065929,0.088737,0.075910,-0.030257
clo:0000374,-0.000659,0.049414,-0.069594,-0.053286,-0.082429,0.044230,0.061466,-0.063945,-0.068006,0.044058,...,-0.095151,-0.092239,0.015987,0.012241,0.007833,-0.082954,0.038649,0.036538,-0.042749,-0.095698
ensembl:ENST00000256362,-0.027396,0.101125,0.044499,0.001286,-0.015335,0.134640,-0.050498,-0.041454,0.069795,-0.076664,...,-0.016087,-0.008311,0.035215,-0.083921,-0.035322,-0.038975,0.007342,0.055210,0.090515,0.043116
so:0000673,-0.017527,0.084457,-0.037265,0.060995,-0.009120,0.018535,0.004754,-0.010168,-0.036380,0.045767,...,-0.011469,-0.052057,-0.006376,-0.008064,-0.030876,-0.098405,0.034433,-0.007673,0.013841,-0.177492
ncbigene:55237,0.082358,-0.078504,0.110313,-0.005730,-0.072022,-0.078058,-0.173017,0.008609,-0.084224,0.073359,...,-0.070185,-0.138352,-0.007950,-0.037352,-0.006233,-0.141211,-0.198813,-0.205973,0.049591,0.062639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
pr:p56678,0.107904,0.010806,-0.008740,0.087995,0.007286,0.038464,0.054555,0.018022,0.021802,-0.122494,...,0.021185,-0.041440,0.068795,0.028714,-0.076069,-0.059464,-0.001187,0.025375,0.002430,-0.006568
pr:p69747,0.026118,-0.018719,-0.065038,-0.167955,-0.036521,0.009081,-0.083593,-0.047875,0.152020,-0.068961,...,-0.004255,-0.027041,0.022365,0.042645,0.051006,-0.151500,-0.073157,0.019740,0.020742,0.053706
pr:p94298,0.175025,0.060768,-0.165581,0.151124,0.122776,-0.199425,-0.207134,0.002505,0.095402,-0.105098,...,-0.016290,0.091887,0.166056,-0.032680,0.149220,-0.145068,-0.176207,0.286030,0.125628,-0.124131
